# Task 1: Movie Search System
## Keyword Search (BM25 & TF-IDF) dan Semantic Search (Vector Database)

Pada notebook ini, kita akan membuat sistem pencarian film dengan dua pendekatan:
1. *Keyword Search*: BM25 dan TF-IDF untuk pencarian berbasis kata kunci
2. *Semantic Search*: Menggunakan embeddings dan vector database (Pinecone & ChromaDB)

## 1. Setup dan Import Libraries

In [3]:
pip install rank_bm25 scikit-learn pandas numpy chromadb pinecone-client google-generativeai python-dotenv tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Keyword Search
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Embedding Model
from sentence_transformers import SentenceTransformer

# Vector Databases
import chromadb
from chromadb.config import Settings
import pinecone

# Load environment variables
load_dotenv()

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Load dan Eksplorasi Dataset

In [3]:
# Load movies dataset
df = pd.read_csv('movies_metadata.csv', low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (45466, 24)

Columns: ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

First few rows:


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [4]:
# Check data info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  str    
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [ ]:

df_clean = df[['id', 'title', 'overview', 'genres', 'release_date', 'vote_average']].copy()

# Remove rows dengan overview kosong
df_clean = df_clean.dropna(subset=['overview', 'title'])

# Parse genres dari JSON string
def parse_genres(genres_str):
    try:
        genres = json.loads(genres_str.replace("'", '"'))
        return ', '.join([g['name'] for g in genres])
    except:
        return ''

df_clean['genres_parsed'] = df_clean['genres'].apply(parse_genres)

df_clean['searchable_text'] = (df_clean['title'].fillna('') + ' ' + 
                                df_clean['overview'].fillna('') + ' ' + 
                                df_clean['genres_parsed'].fillna(''))

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"\nSample data:")
df_clean[['title', 'genres_parsed', 'overview']].head()

Cleaned dataset shape: (44506, 8)

Sample data:


,title,genres_parsed,overview
0,Toy Story,"Animation, Comedy, Family","Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,"Adventure, Fantasy, Family",When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,"Romance, Comedy",A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Comedy, Drama, Romance","Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...


## 3. Keyword Search - BM25

In [6]:
# Prepare corpus for BM25
corpus = df_clean['searchable_text'].tolist()
tokenized_corpus = [doc.lower().split() for doc in corpus]

# Initialize BM25
bm25 = BM25Okapi(tokenized_corpus)

print("✓ BM25 model initialized")

✓ BM25 model initialized


In [7]:
def search_bm25(query, top_k=5):
    """
    Search menggunakan BM25
    """
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    # Get top k indices
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'title': df_clean.iloc[idx]['title'],
            'overview': df_clean.iloc[idx]['overview'],
            'genres': df_clean.iloc[idx]['genres_parsed'],
            'score': scores[idx],
            'release_date': df_clean.iloc[idx]['release_date'],
            'rating': df_clean.iloc[idx]['vote_average']
        })
    
    return results

In [8]:
# Test BM25 Search - Query 1: robot
print("="*80)
print("BM25 Search Results for: 'robot'")
print("="*80)

results = search_bm25('robot', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Score: {result['score']:.2f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:150]}...")

BM25 Search Results for: 'robot'

1. Cody the Robosapien (Score: 11.18)
   Genres: Adventure, Drama, Family, Science Fiction
   Rating: 5.4 | Release: 2013-04-01
   Overview: At Kinetech Labs, an inventor named Allan Topher designs a robot for search and rescue, but when he finds out that the robot will be used for military...

2. Robot Stories (Score: 10.94)
   Genres: Drama, Science Fiction, Romance
   Rating: 3.7 | Release: 2003-01-01
   Overview: Four stories including: "My Robot Baby," in which a couple must care for a robot baby before adopting a human child; "The Robot Fixer," in which a mot...

3. The Robot vs. The Aztec Mummy (Score: 10.39)
   Genres: Science Fiction, Horror
   Rating: 2.8 | Release: 1958-07-17
   Overview: A mad doctor builds a robot in order to steal a valuable Aztec treasure from a tomb guarded by a centuries old living mummy....

4. Eve of Destruction (Score: 10.33)
   Genres: Action, Science Fiction
   Rating: 5.1 | Release: 1991-01-18
   Overview: Eve is

In [9]:
# Test BM25 Search - Query 2: love
print("="*80)
print("BM25 Search Results for: 'love'")
print("="*80)

results = search_bm25('love', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Score: {result['score']:.2f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:150]}...")

BM25 Search Results for: 'love'

1. Out of Love (Score: 4.76)
   Genres: Drama
   Rating: 4.8 | Release: 2016-08-11
   Overview: Out of Love encapsulates the sweltering and devastating dynamics of love in the turbulent relationship between Varya and Nikolai, where genuine love a...

2. Future My Love (Score: 4.55)
   Genres: Documentary, Romance
   Rating: 2.8 | Release: 2012-06-21
   Overview: Directed and narrated by Maja Borg, Future My Love is a unique love story challenging our collective and personal utopias in search of freedom....

3. Love Me (Score: 4.53)
   Genres: Documentary
   Rating: 6.7 | Release: 2014-04-06
   Overview: Love Me follows Western men and Ukrainian women as they embark on an unpredictable and riveting journey in search of love through the modern "mail-ord...

4. The Quiet Love (Score: 4.49)
   Genres: Drama, Romance
   Rating: 6.0 | Release: 2005-11-22
   Overview: Two very different love stories cross each other in warm and magical way. A humanistic and ye

In [10]:
# Test BM25 Search - Query 3: adventure
print("="*80)
print("BM25 Search Results for: 'adventure'")
print("="*80)

results = search_bm25('adventure', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Score: {result['score']:.2f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:150]}...")

BM25 Search Results for: 'adventure'

1. Tarzan's Greatest Adventure (Score: 6.65)
   Genres: Adventure
   Rating: 7.0 | Release: 1959-07-08
   Overview: The greatest adventure of jungle king Tarzan. Four British villains raid a settlement to obtain explosives for use in a diamond mine. In doing so they...

2. Balto II: Wolf Quest (Score: 6.63)
   Genres: Family, Animation, Adventure
   Rating: 6.2 | Release: 2002-02-19
   Overview: Balto and his daughter Aleu embark on a journey of adventure and self discovery....

3. The Poseidon Adventure (Score: 6.45)
   Genres: Action, Adventure
   Rating: 7.3 | Release: 1972-12-01
   Overview: The Poseidon Adventure was one of the first Catastrophe films and began the Disaster Film genre. Director Neame tells the story of a group of people t...

4. Lusers (Score: 6.06)
   Genres: Comedy, Adventure
   Rating: 2.0 | Release: 2015-10-01
   Overview: Three guys from differents countries (Peru, Argentina and Chile) come together in a funny adventure t

## 4. Keyword Search - TF-IDF

In [11]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

# Fit and transform
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print(f"✓ TF-IDF matrix shape: {tfidf_matrix.shape}")

✓ TF-IDF matrix shape: (44506, 5000)


In [12]:
def search_tfidf(query, top_k=5):
    """
    Search menggunakan TF-IDF
    """
    # Transform query
    query_vec = tfidf_vectorizer.transform([query])
    
    # Calculate cosine similarity
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Get top k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'title': df_clean.iloc[idx]['title'],
            'overview': df_clean.iloc[idx]['overview'],
            'genres': df_clean.iloc[idx]['genres_parsed'],
            'score': similarities[idx],
            'release_date': df_clean.iloc[idx]['release_date'],
            'rating': df_clean.iloc[idx]['vote_average']
        })
    
    return results

In [13]:
# Test TF-IDF Search - Query 1: robot
print("="*80)
print("TF-IDF Search Results for: 'robot'")
print("="*80)

results = search_tfidf('robot', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Score: {result['score']:.4f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:150]}...")

TF-IDF Search Results for: 'robot'

1. OMG, I'm a Robot! (Score: 0.7248)
   Genres: Comedy, Science Fiction
   Rating: 10.0 | Release: 2015-08-06
   Overview: A sensitive guy finds out he's... a robot....

2. Cody the Robosapien (Score: 0.6799)
   Genres: Adventure, Drama, Family, Science Fiction
   Rating: 5.4 | Release: 2013-04-01
   Overview: At Kinetech Labs, an inventor named Allan Topher designs a robot for search and rescue, but when he finds out that the robot will be used for military...

3. Robot Stories (Score: 0.6484)
   Genres: Drama, Science Fiction, Romance
   Rating: 3.7 | Release: 2003-01-01
   Overview: Four stories including: "My Robot Baby," in which a couple must care for a robot baby before adopting a human child; "The Robot Fixer," in which a mot...

4. Robot Jox (Score: 0.5658)
   Genres: Science Fiction, Action
   Rating: 5.3 | Release: 1989-10-01
   Overview: 50 years after a nuclear war, the two superpowers handle territorial disputes in a different way. Each

In [14]:
# Test TF-IDF Search - Query 2: love
print("="*80)
print("TF-IDF Search Results for: 'love'")
print("="*80)

results = search_tfidf('love', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Score: {result['score']:.4f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:150]}...")

TF-IDF Search Results for: 'love'

1. Out of Love (Score: 0.5929)
   Genres: Drama
   Rating: 4.8 | Release: 2016-08-11
   Overview: Out of Love encapsulates the sweltering and devastating dynamics of love in the turbulent relationship between Varya and Nikolai, where genuine love a...

2. Love is All: 100 Years of Love & Courtship (Score: 0.5273)
   Genres: Documentary
   Rating: 7.3 | Release: 2014-12-07
   Overview: A magical and moving archive trip through the universal theme of love, from the very first kisses ever caught on film, through the disruption of war t...

3. The Love of Ulysses (Score: 0.5183)
   Genres: 
   Rating: 0.0 | Release: 1984-03-18
   Overview: Directed by Vasilis Vafeas...

4. If You Love (Score: 0.5089)
   Genres: Drama, Comedy
   Rating: 7.5 | Release: 2010-01-06
   Overview: Musiikkielokuva joka kertoo ministerin tyttären ja maahanmuuttajien pojan rosoisen rakkaustarinan....

5. A Cure for Love (Score: 0.4913)
   Genres: Comedy, Adventure, Crime
   Rating:

In [15]:
# Test TF-IDF Search - Query 3: adventure
print("="*80)
print("TF-IDF Search Results for: 'adventure'")
print("="*80)

results = search_tfidf('adventure', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Score: {result['score']:.4f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:150]}...")

TF-IDF Search Results for: 'adventure'

1. Mei and the Kittenbus (Score: 0.4383)
   Genres: Adventure, Animation, Fantasy
   Rating: 7.3 | Release: 2002-01-01
   Overview: Mei has an adventure with a Kittenbus and her relatives. Totoro appears....

2. Green Ice (Score: 0.3718)
   Genres: Adventure, Romance
   Rating: 4.4 | Release: 1981-05-01
   Overview: He wanted adventure...She craved revenge...Emeralds held the answer....

3. The Huns (Score: 0.3708)
   Genres: Action, Adventure
   Rating: 0.0 | Release: 1960-10-05
   Overview: The story of the Queen of the Tartars....

4. The Poseidon Adventure (Score: 0.3655)
   Genres: Action, Adventure
   Rating: 7.3 | Release: 1972-12-01
   Overview: The Poseidon Adventure was one of the first Catastrophe films and began the Disaster Film genre. Director Neame tells the story of a group of people t...

5. Goodbye Pork Pie (Score: 0.3511)
   Genres: Adventure, Comedy
   Rating: 8.5 | Release: 1981-02-05
   Overview: Gerry hires a car in Kaitaia

## 5. Semantic Search - Load Embedding Model

In [ ]:

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("✓ Embedding model loaded successfully")
print(f"   Model: all-MiniLM-L6-v2")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


✓ Embedding model loaded successfully
   Model: all-MiniLM-L6-v2
   Embedding dimension: 384


In [17]:
# Test embedding model
test_sentences = [
    "A robot movie with action",
    "Romantic comedy about love",
    "Adventure in the jungle"
]

test_embeddings = model.encode(test_sentences)
print(f"Test embeddings shape: {test_embeddings.shape}")
print(f"Sample embedding (first 10 values): {test_embeddings[0][:10]}")

Test embeddings shape: (3, 384)
Sample embedding (first 10 values): [-0.07234666 -0.05041654 -0.06787503 -0.04470568 -0.00411086  0.021693
  0.0501724   0.0190766   0.04194644 -0.01869071]


## 6. Semantic Search - ChromaDB (Local Vector Database)

In [20]:
import chromadb
from sentence_transformers import SentenceTransformer

#  Load model embedding
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


In [ ]:

chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create or get collection
try:
    chroma_client.delete_collection(name="movies")
except:
    pass

collection = chroma_client.create_collection(
    name="movies",
    metadata={"hnsw:space": "cosine"}
)

print("✓ ChromaDB collection created")

✓ ChromaDB collection created


In [ ]:

sample_size = 1000
df_sample = df_clean.head(sample_size).copy()

print(f"Generating embeddings for {sample_size} movies...")

# Generate embeddings
embeddings = model.encode(
    df_sample['searchable_text'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"✓ Embeddings generated: {embeddings.shape}")

Generating embeddings for 1000 movies...


Batches: 100%|██████████| 32/32 [00:11<00:00,  2.80it/s]

✓ Embeddings generated: (1000, 384)


In [24]:
# Add to ChromaDB
print("Adding data to ChromaDB...")

collection.add(
    embeddings=embeddings.tolist(),
    documents=df_sample['searchable_text'].tolist(),
    metadatas=[
        {
            'title': row['title'],
            'genres': row['genres_parsed'],
            'overview': row['overview'][:500],  # Limit length
            'release_date': str(row['release_date']),
            'rating': str(row['vote_average'])
        }
        for _, row in df_sample.iterrows()
    ],
    ids=[f"movie_{i}" for i in range(len(df_sample))]
)

print(f"✓ Added {len(df_sample)} movies to ChromaDB")

Adding data to ChromaDB...
✓ Added 1000 movies to ChromaDB


In [25]:
def search_chromadb(query, top_k=5):
    """
    Semantic search menggunakan ChromaDB
    """
    # Encode query
    query_embedding = model.encode([query])
    
    # Search in ChromaDB
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    
    formatted_results = []
    for i in range(len(results['ids'][0])):
        metadata = results['metadatas'][0][i]
        formatted_results.append({
            'title': metadata['title'],
            'overview': metadata['overview'],
            'genres': metadata['genres'],
            'distance': results['distances'][0][i],
            'release_date': metadata['release_date'],
            'rating': metadata['rating']
        })
    
    return formatted_results

In [26]:
# Test ChromaDB Semantic Search - Query 1
print("="*80)
print("ChromaDB Semantic Search: 'Adventure movie with characters named Judy and Peter'")
print("="*80)

results = search_chromadb('Adventure movie with characters named Judy and Peter', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Distance: {result['distance']:.4f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:200]}...")

ChromaDB Semantic Search: 'Adventure movie with characters named Judy and Peter'

1. The Wizard of Oz (Distance: 0.5129)
   Genres: Adventure, Family, Fantasy
   Rating: 7.4 | Release: 1939-08-15
   Overview: Young Dorothy finds herself in a magical world where she makes friends with a lion, a scarecrow and a tin man as they make their way along the yellow brick road to talk with the Wizard and ask for the...

2. Jumanji (Distance: 0.5321)
   Genres: Adventure, Fantasy, Family
   Rating: 6.9 | Release: 1995-12-15
   Overview: When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into t...

3. Pete's Dragon (Distance: 0.5334)
   Genres: Fantasy, Animation, Comedy, Family
   Rating: 6.4 | Release: 1977-11-03
   Overview: Pete, a young orphan, runs away to a Maine fishing town with his best friend a lovable, sometimes invisible dragon named Elliott! W

In [27]:
# Test ChromaDB Semantic Search - Query 2
print("="*80)
print("ChromaDB Semantic Search: 'Movies with scenes in New York City'")
print("="*80)

results = search_chromadb('Movies with scenes in New York City', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Distance: {result['distance']:.4f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:200]}...")

ChromaDB Semantic Search: 'Movies with scenes in New York City'

1. City Hall (Distance: 0.4962)
   Genres: Drama, Thriller
   Rating: 6.0 | Release: 1996-02-16
   Overview: The accidental shooting of a boy in New York leads to an investigation by the Deputy Mayor, and unexpectedly far-reaching consequences....

2. Joe's Apartment (Distance: 0.4973)
   Genres: Fantasy, Comedy, Music
   Rating: 5.0 | Release: 1996-07-26
   Overview: A nice guy has just moved to New York and discovers that he must share his run-down apartment with a couple thousand singing, dancing cockroaches....

3. Manhattan Murder Mystery (Distance: 0.5081)
   Genres: Comedy, Mystery
   Rating: 7.1 | Release: 1993-08-18
   Overview: A middle-aged couple suspects foul play when their neighbor's wife suddenly drops dead....

4. Martin Lawrence: You So Crazy (Distance: 0.5109)
   Genres: Comedy
   Rating: 4.5 | Release: 1994-04-27
   Overview: Stand up comedy by Martin Lawrence, filmed in the Majestic Theater in New Yor

In [28]:
# Test ChromaDB Semantic Search - Query 3
print("="*80)
print("ChromaDB Semantic Search: 'movie about poet Arthur Rimbaud and Paul Verlaine relationship'")
print("="*80)

results = search_chromadb('movie about poet Arthur Rimbaud and Paul Verlaine relationship', top_k=5)
for i, result in enumerate(results, 1):
    print(f"\n{i}. {result['title']} (Distance: {result['distance']:.4f})")
    print(f"   Genres: {result['genres']}")
    print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
    print(f"   Overview: {result['overview'][:200]}...")

ChromaDB Semantic Search: 'movie about poet Arthur Rimbaud and Paul Verlaine relationship'

1. Total Eclipse (Distance: 0.3648)
   Genres: Drama, Romance
   Rating: 6.5 | Release: 1995-11-02
   Overview: Young, wild poet Arthur Rimbaud and his mentor Paul Verlaine engage in a fierce, forbidden romance while feeling the effects of a hellish artistic lifestyle....

2. Delta of Venus (Distance: 0.5923)
   Genres: Drama, Romance
   Rating: 3.3 | Release: 1995-06-09
   Overview: A struggling American writer and a fellow American expatriate begin a sordid affair among the chaos and discord of 1940 Paris, France on the brink of World War II....

3. Honeymoon (Distance: 0.6004)
   Genres: Comedy
   Rating: 0.0 | Release: 1996-04-11
   Overview: German Comedy...

4. Poetic Justice (Distance: 0.6188)
   Genres: Drama, Romance
   Rating: 6.9 | Release: 1993-07-23
   Overview: In this film, we see the world through the eyes of main character Justice, a young African-American poet. A mail carrier i

## 7. Semantic Search - Pinecone (Cloud Vector Database)

In [32]:
from pinecone import Pinecone, ServerlessSpec
import os

In [33]:
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY', '')

if PINECONE_API_KEY:
    try:
        # 1. Inisialisasi instance Pinecone versi terbaru
        pc = Pinecone(api_key=PINECONE_API_KEY)
        
        index_name = 'movies-search'
        
        # 2. Cek apakah index sudah ada
        if index_name not in pc.list_indexes().names():
            print(f"Creating index '{index_name}'...")
            pc.create_index(
                name=index_name,
                dimension=384,  # dimensi dari model all-MiniLM-L6-v2
                metric='cosine',
                spec=ServerlessSpec(
                    cloud='aws',
                    region='us-east-1' # Sesuai dengan setting di dashboard Anda
                )
            )
            print(f"✓ Created Pinecone index: {index_name}")
        else:
            print(f"✓ Connected to existing Pinecone index: {index_name}")
            
        # 3. Connect ke index
        pinecone_index = pc.Index(index_name)
        
    except Exception as e:
        print(f"⚠ Pinecone initialization failed: {e}")
        print("   Skipping Pinecone demo. Please set PINECONE_API_KEY in .env")
        pinecone_index = None
else:
    print("⚠ PINECONE_API_KEY not found in .env")
    print("   Skipping Pinecone demo. ChromaDB works perfectly for local development!")
    pinecone_index = None

✓ Connected to existing Pinecone index: movies-search


In [ ]:

if pinecone_index:
    print("Uploading to Pinecone...")
    
    # Prepare data for upsert
    vectors_to_upsert = []
    
    for i, (embedding, row) in enumerate(zip(embeddings, df_sample.itertuples())):
        vectors_to_upsert.append({
            'id': f'movie_{i}',
            'values': embedding.tolist(),
            'metadata': {
                'title': row.title,
                'genres': row.genres_parsed,
                'overview': row.overview[:500],
                'release_date': str(row.release_date),
                'rating': float(row.vote_average) if pd.notna(row.vote_average) else 0.0
            }
        })
    
    # Upsert in batches
    batch_size = 100
    for i in range(0, len(vectors_to_upsert), batch_size):
        batch = vectors_to_upsert[i:i+batch_size]
        pinecone_index.upsert(vectors=batch)
        print(f"   Uploaded {min(i+batch_size, len(vectors_to_upsert))}/{len(vectors_to_upsert)} vectors")
    
    print(f"✓ Uploaded {len(vectors_to_upsert)} movies to Pinecone")
else:
    print("Skipping Pinecone upload (not configured)")

Uploading to Pinecone...
   Uploaded 100/1000 vectors
   Uploaded 200/1000 vectors
   Uploaded 300/1000 vectors
   Uploaded 400/1000 vectors
   Uploaded 500/1000 vectors
   Uploaded 600/1000 vectors
   Uploaded 700/1000 vectors
   Uploaded 800/1000 vectors
   Uploaded 900/1000 vectors
   Uploaded 1000/1000 vectors
✓ Uploaded 1000 movies to Pinecone


In [35]:
def search_pinecone(query, top_k=5):
    """
    Semantic search menggunakan Pinecone
    """
    if not pinecone_index:
        return [{"error": "Pinecone not configured"}]
    
    # Encode query
    query_embedding = model.encode([query])
    
    # Search in Pinecone
    results = pinecone_index.query(
        vector=query_embedding[0].tolist(),
        top_k=top_k,
        include_metadata=True
    )
    
    formatted_results = []
    for match in results['matches']:
        metadata = match['metadata']
        formatted_results.append({
            'title': metadata['title'],
            'overview': metadata['overview'],
            'genres': metadata['genres'],
            'score': match['score'],
            'release_date': metadata['release_date'],
            'rating': metadata['rating']
        })
    
    return formatted_results

In [36]:
# Test Pinecone Semantic Search
if pinecone_index:
    print("="*80)
    print("Pinecone Semantic Search: 'Adventure movie with characters named Judy and Peter'")
    print("="*80)
    
    results = search_pinecone('Adventure movie with characters named Judy and Peter', top_k=5)
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['title']} (Score: {result['score']:.4f})")
        print(f"   Genres: {result['genres']}")
        print(f"   Rating: {result['rating']} | Release: {result['release_date']}")
        print(f"   Overview: {result['overview'][:200]}...")
else:
    print("Pinecone not configured - skipping test")

Pinecone Semantic Search: 'Adventure movie with characters named Judy and Peter'

1. The Wizard of Oz (Score: 0.4877)
   Genres: Adventure, Family, Fantasy
   Rating: 7.4 | Release: 1939-08-15
   Overview: Young Dorothy finds herself in a magical world where she makes friends with a lion, a scarecrow and a tin man as they make their way along the yellow brick road to talk with the Wizard and ask for the...

2. Jumanji (Score: 0.4680)
   Genres: Adventure, Fantasy, Family
   Rating: 6.9 | Release: 1995-12-15
   Overview: When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into t...

3. Pete's Dragon (Score: 0.4673)
   Genres: Fantasy, Animation, Comedy, Family
   Rating: 6.4 | Release: 1977-11-03
   Overview: Pete, a young orphan, runs away to a Maine fishing town with his best friend a lovable, sometimes invisible dragon named Elliott! When they 

## 8. Comparison: Keyword vs Semantic Search

In [38]:
def compare_searches(query):
    """
    Compare hasil dari berbagai metode pencarian
    """
    print("="*100)
    print(f"COMPARISON FOR QUERY: '{query}'")
    print("="*100)
    
    # BM25
    print("\n BM25 (Keyword Search):")
    print("-" * 100)
    bm25_results = search_bm25(query, top_k=3)
    for i, r in enumerate(bm25_results, 1):
        print(f"{i}. {r['title']} (Score: {r['score']:.2f})")
    
    # TF-IDF
    print("\n TF-IDF (Keyword Search):")
    print("-" * 100)
    tfidf_results = search_tfidf(query, top_k=3)
    for i, r in enumerate(tfidf_results, 1):
        print(f"{i}. {r['title']} (Score: {r['score']:.4f})")
    
    # ChromaDB
    print("\n ChromaDB (Semantic Search):")
    print("-" * 100)
    chroma_results = search_chromadb(query, top_k=3)
    for i, r in enumerate(chroma_results, 1):
        print(f"{i}. {r['title']} (Distance: {r['distance']:.4f})")
    
    print("\n" + "="*100)

In [39]:
# Test berbagai query
test_queries = [
    "robot",
    "love",
    "adventure",
    "Adventure movie with characters named Judy and Peter",
    "Movies with scenes in New York City"
]

for query in test_queries:
    compare_searches(query)
    print("\n\n")

COMPARISON FOR QUERY: 'robot'

 BM25 (Keyword Search):
----------------------------------------------------------------------------------------------------
1. Cody the Robosapien (Score: 11.18)
2. Robot Stories (Score: 10.94)
3. The Robot vs. The Aztec Mummy (Score: 10.39)

 TF-IDF (Keyword Search):
----------------------------------------------------------------------------------------------------
1. OMG, I'm a Robot! (Score: 0.7248)
2. Cody the Robosapien (Score: 0.6799)
3. Robot Stories (Score: 0.6484)

 ChromaDB (Semantic Search):
----------------------------------------------------------------------------------------------------
1. Solo (Distance: 0.5906)
2. 2001: A Space Odyssey (Distance: 0.6520)
3. Pinocchio (Distance: 0.6576)




COMPARISON FOR QUERY: 'love'

 BM25 (Keyword Search):
----------------------------------------------------------------------------------------------------
1. Out of Love (Score: 4.76)
2. Future My Love (Score: 4.55)
3. Love Me (Score: 4.53)

 TF-IDF (

## 9. Kesimpulan

### Keyword Search (BM25 & TF-IDF):
- Cocok untuk exact match dan keyword-based queries
- Sangat cepat dan efisien
- Tidak memahami semantik/konteks
- Sulit handle sinonim atau parafrase

### Semantic Search (Vector Database):
- Memahami konteks dan makna query
- Dapat handle complex queries
- Menemukan hasil relevan meskipun kata berbeda
- Lebih lambat (perlu embedding)
- Membutuhkan lebih banyak resource

### Best Practice:
Gunakan *Hybrid Search* - kombinasi keyword dan semantic search untuk hasil optimal!